# Feature Engineering — Early Warning des demandes d'asile

## Objectif

Construire **un seul dataset propre pour le Machine Learning Early Warning**.

L'unité d'analyse retenue est :

**pays d'origine (`coo_id`) × pays d'asile (`coa_id`) × année (`year`)**

Le notebook :
- agrège les anciennes sous-catégories procédurales au niveau corridor ;
- construit des features disponibles à **T ou avant T** ;
- conserve `applied_t1` uniquement comme **outcome futur** pour construire la cible dans le notebook ML ;
- ne construit **aucun score de criticité** ici ;
- ne réalise **aucune imputation statistique** avant le split ML ;
- ne supprime pas silencieusement les anomalies.

> Le scoring, la définition de la cible d'escalade, la calibration des probabilités et les niveaux Normal / Attention / High / Critical appartiennent au notebook Machine Learning.


In [35]:
# ============================================================
# 1. IMPORTS ET PARAMÈTRES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INPUT_PATH = Path("../data/consolidated/dataset_demandes_consolide_detail.csv")
OUTPUT_PATH = Path("../data/feature_engineering_outputs/dataset_ml_features.csv")
DQ_PATH = Path("../data/feature_engineering_outputs/data_quality_feature_engineering.csv")
AUDIT_PATH = Path("../data/feature_engineering_outputs/audit_feature_engineering.csv")
COUNTRIES_PATH = Path("../data/raw/countries.csv")

CORRIDOR = ["coo_id", "coa_id"]
KEY = CORRIDOR + ["year"]


## 2. Chargement et contrôles minimaux

Les colonnes indispensables sont `coo_id`, `coa_id`, `year` et `applied`.

Le dataset source n'est jamais modifié sur disque.


In [36]:
df_raw = pd.read_csv(INPUT_PATH)
df = df_raw.copy()

required = {"coo_id", "coa_id", "year", "applied"}
missing_required = required - set(df.columns)

assert not missing_required, (
    f"Colonnes obligatoires absentes : {sorted(missing_required)}"
)

print("Dimensions source :", df.shape)
display(df.head())


Dimensions source : (120597, 50)


,_row_id,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,app_type,dec_level,app_pc,applied,_demand_row_id,has_asylum_application_data,decisions_recognized,decisions_other,decisions_rejected,decisions_closed,decisions_total,decisions_source_rows,recognition_rate,rejection_rate,has_decisions_data,returned_refugees,resettlement,naturalisation,returned_idps,solutions_source_rows,has_solutions_data,origin_iso,origin_iso2,origin_country,origin_country_fr,origin_region,origin_region_fr,origin_major_area,origin_major_area_fr,asylum_iso,asylum_iso2,asylum_country,asylum_country_fr,asylum_region,asylum_region_fr,asylum_major_area,asylum_major_area_fr,idmc_total_origin,has_idmc_data
0,0,2006,2,Afghanistan,AFG,AFG,11,Australia,AUL,AUS,G,A,AR,C,14,0,1,20.0000,0.0000,0.0000,5.0000,25.0000,1.0000,0.8000,0.0000,1,NaN,"2,048.0000",NaN,NaN,1.0000,1,AFG,AF,Afghanistan,Afghanistan,Southern Asia,Sud de l'Asie centrale,Asia,Asie,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
1,1,2006,3,Albania,ALB,ALB,11,Australia,AUL,AUS,G,A,AR,C,21,1,1,5.0000,0.0000,13.0000,0.0000,18.0000,1.0000,0.2778,0.7222,1,NaN,NaN,NaN,NaN,NaN,0,ALB,AL,Albania,Albanie,Southern Europe,Europe australe,Europe,Europe,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
2,2,2006,4,Algeria,ALG,DZA,11,Australia,AUL,AUS,G,A,AR,C,5,2,1,0.0000,0.0000,5.0000,0.0000,5.0000,1.0000,0.0000,1.0000,1,NaN,NaN,NaN,NaN,NaN,0,DZA,DZ,Algeria,Alg,Northern Africa,Afrique septentrionale,Africa,Afrique,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
3,3,2006,8,Egypt,ARE,EGY,11,Australia,AUL,AUS,G,A,AR,C,38,3,1,28.0000,0.0000,20.0000,0.0000,48.0000,1.0000,0.5833,0.4167,1,NaN,25.0000,NaN,NaN,1.0000,1,EGY,EG,Egypt,Egypte,Northern Africa,Afrique septentrionale,Africa,Afrique,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
4,4,2006,14,Bahrain,BAH,BHR,11,Australia,AUL,AUS,G,A,AR,C,11,4,1,11.0000,0.0000,0.0000,0.0000,11.0000,1.0000,1.0000,0.0000,1,NaN,NaN,NaN,NaN,NaN,0,BHR,BH,Bahrain,Bahre,Western Asia,Asie occidentale,Asia,Asie,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0


In [37]:
# ============================================================
# 3. STANDARDISATION ET VALIDITÉ
# ============================================================

audit = []

# Identifiants : chaînes propres, sans imposer un ordre numérique.
for col in ["coo_id", "coa_id"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("UNKNOWN")
    )

# Contexte textuel éventuellement disponible.
for col in ["origin_region", "asylum_region"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("UNKNOWN")
        )

# Conversion explicite des variables numériques.
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["applied"] = pd.to_numeric(df["applied"], errors="coerce")

invalid_year = df["year"].isna()
invalid_applied = df["applied"].isna() | df["applied"].lt(0)

audit.append({
    "control": "invalid_year",
    "n_rows": int(invalid_year.sum()),
    "action": "conserved_and_flagged"
})
audit.append({
    "control": "invalid_applied",
    "n_rows": int(invalid_applied.sum()),
    "action": "conserved_and_flagged"
})

df["invalid_year"] = invalid_year.astype("Int8")
df["invalid_applied"] = invalid_applied.astype("Int8")

print("Années invalides :", int(invalid_year.sum()))
print("Volumes applied invalides :", int(invalid_applied.sum()))


Années invalides : 0
Volumes applied invalides : 0


In [38]:
# ============================================================
# 3 RÉFÉRENTIEL PAYS
# ============================================================

# Le référentiel pays utilise `id` comme clé technique.
# Les noms sont utilisés pour la restitution métier.
# Les IDs restent conservés comme clés de jointure et de traçabilité.

countries = pd.read_csv(COUNTRIES_PATH)

required_country_cols = {"id", "name"}
missing_country_cols = required_country_cols - set(countries.columns)
assert not missing_country_cols, (
    f"Colonnes absentes du référentiel pays : {sorted(missing_country_cols)}"
)

country_lookup = countries[["id", "name"]].copy()
country_lookup["id"] = pd.to_numeric(country_lookup["id"], errors="coerce")

assert country_lookup["id"].notna().all(), (
    "Le référentiel pays contient des IDs non numériques."
)

country_lookup["id"] = country_lookup["id"].astype(int)

# Un ID doit correspondre à un seul pays.
country_conflicts = (
    country_lookup.groupby("id")["name"]
    .nunique(dropna=False)
    .gt(1)
)
assert not country_conflicts.any(), (
    "Plusieurs noms sont associés au même ID pays."
)

country_lookup = (
    country_lookup
    .drop_duplicates("id")
    .rename(columns={"id": "country_id", "name": "country_name"})
)

# Contrôle de couverture avant enrichissement.
coo_ids = pd.to_numeric(df["coo_id"], errors="coerce")
coa_ids = pd.to_numeric(df["coa_id"], errors="coerce")

coo_unmatched = sorted(
    set(coo_ids.dropna().astype(int))
    - set(country_lookup["country_id"])
)
coa_unmatched = sorted(
    set(coa_ids.dropna().astype(int))
    - set(country_lookup["country_id"])
)

audit.append({
    "control": "coo_country_mapping_unmatched_ids",
    "n_rows": len(coo_unmatched),
    "action": "logged_and_preserved"
})
audit.append({
    "control": "coa_country_mapping_unmatched_ids",
    "n_rows": len(coa_unmatched),
    "action": "logged_and_preserved"
})

print("IDs COO sans correspondance :", len(coo_unmatched))
print("IDs COA sans correspondance :", len(coa_unmatched))


IDs COO sans correspondance : 0
IDs COA sans correspondance : 0


## 4. Population exploitable pour les calculs temporels

Les lignes invalides sont **journalisées et conservées dans la source**, mais elles ne peuvent pas participer aux agrégations temporelles.

Le notebook crée donc une vue de travail valide sans supprimer silencieusement les anomalies.


In [39]:
valid_mask = (
    df["year"].notna()
    & df["applied"].notna()
    & df["applied"].ge(0)
    & df["coo_id"].ne("UNKNOWN")
    & df["coa_id"].ne("UNKNOWN")
) 

work = df.loc[valid_mask].copy()
work["year"] = work["year"].astype(int)

audit.append({
    "control": "rows_excluded_from_temporal_computation",
    "n_rows": int((~valid_mask).sum()),
    "action": "logged_not_silently_deleted"
})

print("Lignes source :", len(df))
print("Lignes utilisables pour FE :", len(work))
print("Lignes journalisées hors calcul :", int((~valid_mask).sum()))


Lignes source : 120597
Lignes utilisables pour FE : 120597
Lignes journalisées hors calcul : 0


# 5. Agrégation au niveau corridor

L'ancien segment procédural est remplacé par :

**`coo_id × coa_id × year`**

`applied` est sommé sur les sous-catégories procédurales.

Cette agrégation est réalisée **avant** la création des lags : on ne somme jamais des lags déjà calculés.


In [40]:
corridor = (
    work.groupby(KEY, dropna=False, as_index=False)
    .agg(applied=("applied", "sum"))
)

duplicates_after_aggregation = corridor.duplicated(KEY, keep=False).sum()
assert duplicates_after_aggregation == 0

print("Dimensions corridor × année :", corridor.shape)
print("Corridors uniques :", corridor[CORRIDOR].drop_duplicates().shape[0])
print("Période :", corridor["year"].min(), "-", corridor["year"].max())
display(corridor.head())


Dimensions corridor × année : (86234, 4)
Corridors uniques : 8468
Période : 2000 - 2025


,coo_id,coa_id,year,applied
0,10,109,2019,5
1,10,109,2021,5
2,10,109,2022,10
3,10,11,2000,19
4,10,11,2001,10


In [41]:
# ============================================================
# ENRICHISSEMENT DES CORRIDORS AVEC LES NOMS DE PAYS
# ============================================================

# Nettoyage des colonnes existantes si la cellule est réexécutée
cols_to_drop = [c for c in corridor.columns if "coo_name" in c or "coa_name" in c]
if cols_to_drop:
    corridor = corridor.drop(columns=cols_to_drop)

# Les jointures sont LEFT pour ne perdre aucun corridor.
coo_names = country_lookup.rename(
    columns={"country_id": "coo_id", "country_name": "coo_name"}
)
coa_names = country_lookup.rename(
    columns={"country_id": "coa_id", "country_name": "coa_name"}
)

# Conversion des clés en string pour correspondre au type de corridor
coo_names["coo_id"] = coo_names["coo_id"].astype(str)
coa_names["coa_id"] = coa_names["coa_id"].astype(str)

corridor = corridor.merge(
    coo_names,
    on="coo_id",
    how="left",
    validate="many_to_one"
)

corridor = corridor.merge(
    coa_names,
    on="coa_id",
    how="left",
    validate="many_to_one"
)

corridor["coo_name"] = corridor["coo_name"].fillna("UNKNOWN")
corridor["coa_name"] = corridor["coa_name"].fillna("UNKNOWN")

# Contrôle : l'enrichissement ne doit pas modifier la volumétrie.
print("Corridors après enrichissement pays :", len(corridor))
display(corridor[["coo_id", "coo_name", "coa_id", "coa_name", "year", "applied"]].head())

Corridors après enrichissement pays : 86234


,coo_id,coo_name,coa_id,coa_name,year,applied
0,10,Armenia,109,Lebanon,2019,5
1,10,Armenia,109,Lebanon,2021,5
2,10,Armenia,109,Lebanon,2022,10
3,10,Armenia,11,Australia,2000,19
4,10,Armenia,11,Australia,2001,10


## 6. Contexte géographique

Les régions sont des attributs contextuels du pays.  
On vérifie qu'un même identifiant pays-année n'est pas associé à plusieurs régions avant la jointure.


In [42]:
def attach_context(base, source, id_col, context_col):
    if context_col not in source.columns:
        base[context_col] = "UNKNOWN"
        return base

    lookup = (
        source[[id_col, "year", context_col]]
        .drop_duplicates()
        .copy()
    )

    conflicts = (
        lookup.groupby([id_col, "year"], dropna=False)[context_col]
        .nunique(dropna=False)
        .gt(1)
    )

    n_conflicts = int(conflicts.sum())
    audit.append({
        "control": f"{context_col}_conflicts",
        "n_rows": n_conflicts,
        "action": "stop_if_conflict"
    })

    assert n_conflicts == 0, (
        f"Conflits détectés pour {context_col}. "
        "Corriger le référentiel avant de poursuivre."
    )

    lookup = lookup.drop_duplicates([id_col, "year"])

    return base.merge(
        lookup,
        on=[id_col, "year"],
        how="left",
        validate="many_to_one"
    )

corridor = attach_context(
    corridor, work, "coo_id", "origin_region"
)
corridor = attach_context(
    corridor, work, "coa_id", "asylum_region"
)

for col in ["origin_region", "asylum_region"]:
    corridor[col] = corridor[col].fillna("UNKNOWN")


# 7. Lags calendaires exacts

`lag1` signifie **année T−1 exacte**, pas simplement « ligne précédente ».

Si 2022 manque, la valeur 2021 ne devient pas artificiellement le lag1 de 2023.


In [43]:
def add_exact_lag(data, keys, year_col, value_col, lag, output_col):
    lookup = data[keys + [year_col, value_col]].copy()
    lookup[year_col] = lookup[year_col] + lag
    lookup = lookup.rename(columns={value_col: output_col})

    return data.merge(
        lookup,
        on=keys + [year_col],
        how="left",
        validate="one_to_one"
    )

corridor = add_exact_lag(
    corridor, CORRIDOR, "year", "applied", 1, "applied_lag1"
)
corridor = add_exact_lag(
    corridor, CORRIDOR, "year", "applied", 2, "applied_lag2"
)


# 8. Features de dynamique Early Warning

Les features sont volontairement limitées à des signaux interprétables :

- niveau courant ;
- variation récente ;
- accélération ;
- niveau historique ;
- déviation par rapport au niveau récent ;
- persistance de la hausse.


In [44]:
# ============================================================
# 8. FEATURES DE DYNAMIQUE EARLY WARNING
# ============================================================

# Variation entre T et T-1.
corridor["absolute_change_past_1"] = (
    corridor["applied"] - corridor["applied_lag1"]
)

corridor["growth_past_1"] = np.where(
    corridor["applied_lag1"].gt(0),
    (corridor["applied"] - corridor["applied_lag1"]) / corridor["applied_lag1"],
    np.nan
)

# Variation entre T-1 et T-2.
growth_past_2 = np.where(
    corridor["applied_lag2"].gt(0),
    (corridor["applied_lag1"] - corridor["applied_lag2"]) / corridor["applied_lag2"],
    np.nan
)
growth_past_2 = pd.Series(growth_past_2, index=corridor.index)

# Accélération de la dynamique :
# croissance récente - croissance de la période précédente.
corridor["acceleration"] = (
    corridor["growth_past_1"] - growth_past_2
)

# Indicateur de persistance :
# 1 = deux hausses consécutives observées
# 0 = pas deux hausses consécutives
# <NA> = historique insuffisant pour calculer l'indicateur.
has_history = (
    corridor["applied_lag1"].notna()
    & corridor["applied_lag2"].notna()
)

two_increases = (
    corridor["growth_past_1"].gt(0)
    & growth_past_2.gt(0)
)

corridor["two_consecutive_increases"] = (
    two_increases.where(has_history)
    .astype("Int8")
)

display(
    corridor[
        [
            "coo_id", "coa_id", "year",
            "applied", "applied_lag1", "applied_lag2",
            "growth_past_1", "absolute_change_past_1",
            "acceleration", "two_consecutive_increases"
        ]
    ].head(10)
)


,coo_id,coa_id,year,applied,applied_lag1,applied_lag2,growth_past_1,absolute_change_past_1,acceleration,two_consecutive_increases
0,10,109,2019,5,NaN,NaN,NaN,NaN,NaN,<NA>
1,10,109,2021,5,NaN,5.0000,NaN,NaN,NaN,<NA>
2,10,109,2022,10,5.0000,NaN,1.0000,5.0000,NaN,<NA>
3,10,11,2000,19,NaN,NaN,NaN,NaN,NaN,<NA>
4,10,11,2001,10,19.0000,NaN,-0.4737,-9.0000,NaN,<NA>
5,10,11,2002,10,10.0000,19.0000,0.0000,0.0000,0.4737,0
6,10,11,2003,5,10.0000,10.0000,-0.5000,-5.0000,-0.5000,0
7,10,11,2005,5,NaN,5.0000,NaN,NaN,NaN,<NA>
8,10,11,2013,5,NaN,NaN,NaN,NaN,NaN,<NA>
9,10,11,2023,7,NaN,NaN,NaN,NaN,NaN,<NA>


# 9. Pression origine, pression pays d'asile et poids du corridor

Ces variables replacent le corridor dans son environnement à T.


In [45]:
corridor["origin_applied_total_t"] = (
    corridor.groupby(["coo_id", "year"], dropna=False)["applied"]
    .transform("sum")
)

corridor["asylum_applied_total_t"] = (
    corridor.groupby(["coa_id", "year"], dropna=False)["applied"]
    .transform("sum")
)

corridor["segment_share_origin_t"] = np.where(
    corridor["origin_applied_total_t"].gt(0),
    corridor["applied"] / corridor["origin_applied_total_t"],
    np.nan
)

corridor["segment_share_asylum_t"] = np.where(
    corridor["asylum_applied_total_t"].gt(0),
    corridor["applied"] / corridor["asylum_applied_total_t"],
    np.nan
)


# 10. IDMC T−1

L'IDMC reste un **signal de volume de déplacement interne**, pas une explication de la cause du déplacement.

Pour éviter de compter plusieurs fois une même valeur IDMC répétée sur plusieurs lignes procédurales, on construit d'abord un référentiel **origine × année**, puis on le décale à T−1.

Si plusieurs valeurs différentes existent pour un même pays d'origine et une même année, le notebook s'arrête : le conflit doit être audité.


In [46]:
if "idmc_total_origin_lag1" in work.columns:
    # Cas où la source contient déjà une variable correctement retardée.
    idmc_source_col = "idmc_total_origin_lag1"
    idmc_lookup = work[
        ["coo_id", "year", idmc_source_col]
    ].copy()
    idmc_lookup[idmc_source_col] = pd.to_numeric(
        idmc_lookup[idmc_source_col], errors="coerce"
    )

    conflicts = (
        idmc_lookup.dropna(subset=[idmc_source_col])
        .groupby(["coo_id", "year"])[idmc_source_col]
        .nunique()
        .gt(1)
    )
    assert int(conflicts.sum()) == 0, (
        "Conflits IDMC T-1 détectés pour un même pays-année."
    )

    idmc_lookup = (
        idmc_lookup
        .dropna(subset=[idmc_source_col])
        .drop_duplicates(["coo_id", "year"])
    )

    corridor = corridor.merge(
        idmc_lookup,
        on=["coo_id", "year"],
        how="left",
        validate="many_to_one"
    )

elif "idmc_total_origin" in work.columns:
    idmc = work[["coo_id", "year", "idmc_total_origin"]].copy()
    idmc["idmc_total_origin"] = pd.to_numeric(
        idmc["idmc_total_origin"], errors="coerce"
    )

    conflicts = (
        idmc.dropna(subset=["idmc_total_origin"])
        .groupby(["coo_id", "year"])["idmc_total_origin"]
        .nunique()
        .gt(1)
    )
    assert int(conflicts.sum()) == 0, (
        "Conflits IDMC détectés pour un même pays-année."
    )

    idmc = (
        idmc.dropna(subset=["idmc_total_origin"])
        .drop_duplicates(["coo_id", "year"])
    )

    # La valeur observée en année S doit rejoindre l'année S+1 comme lag1.
    idmc["year"] = idmc["year"] + 1
    idmc = idmc.rename(
        columns={"idmc_total_origin": "idmc_total_origin_lag1"}
    )

    corridor = corridor.merge(
        idmc,
        on=["coo_id", "year"],
        how="left",
        validate="many_to_one"
    )

else:
    corridor["idmc_total_origin_lag1"] = np.nan
    print("ATTENTION : aucune variable IDMC disponible dans la source.")


# 11. Outcome futur `applied_t1`

`applied_t1` est conservé uniquement pour permettre au notebook ML de construire la cible d'escalade.

Il ne fait **jamais** partie des features.


In [47]:
# ============================================================
# 11. OUTCOME FUTUR applied_t1
# ============================================================

# Une valeur observée en année S devient la cible de S-1.
# Exemple : les demandes 2024 deviennent applied_t1 pour 2023.
target_lookup = corridor[KEY + ["applied"]].copy()

target_lookup["year"] = target_lookup["year"] - 1
target_lookup = target_lookup.rename(
    columns={"applied": "applied_t1"}
)

corridor = corridor.merge(
    target_lookup,
    on=KEY,
    how="left",
    validate="one_to_one"
)

# Règle métier validée :
# si un corridor est absent en T+1, cela signifie 0 demande.
# Cette règle ne s'applique que lorsque T+1 est dans la période
# effectivement couverte par la source.
max_source_year = int(corridor["year"].max())

target_observable = corridor["year"] < max_source_year

missing_t1_before_zero = (
    target_observable
    & corridor["applied_t1"].isna()
)

corridor.loc[missing_t1_before_zero, "applied_t1"] = 0

print("Année maximale observée dans la source :", max_source_year)
print(
    "Corridors T+1 absents interprétés comme 0 :",
    int(missing_t1_before_zero.sum())
)
print(
    "Lignes avec outcome T+1 observable :",
    int(target_observable.sum())
)
print(
    "Lignes encore sans outcome T+1 :",
    int(corridor["applied_t1"].isna().sum())
)
print(
    "Taux labellisable :",
    f"{corridor['applied_t1'].notna().mean():.2%}"
)


Année maximale observée dans la source : 2025
Corridors T+1 absents interprétés comme 0 : 15310
Lignes avec outcome T+1 observable : 82267
Lignes encore sans outcome T+1 : 3967
Taux labellisable : 95.40%


# 12. Feature set final

Le Feature Engineering s'arrête ici.

Les variables futures nécessaires à la définition de la criticité seront calculées dans le notebook ML, puis strictement exclues de `X`.


In [55]:
FEATURES = [
    # Temps et contexte.
    "year",
    "coo_id",
    "coa_id",
    "coa_name",
    "coo_name",
    "origin_region",
    "asylum_region",

    # Pression courante et historique.
    "applied",
    "applied_lag1",
    "applied_lag2",

    # Dynamique Early Warning.
    "growth_past_1",
    "absolute_change_past_1",
    "acceleration",
    "two_consecutive_increases",

    # Pression et poids du corridor.
    "origin_applied_total_t",
    "asylum_applied_total_t",
    "segment_share_origin_t",
    "segment_share_asylum_t",

    # Contexte externe.
    "idmc_total_origin_lag1",
]

FEATURES = [c for c in FEATURES if c in corridor.columns]
TARGET_SUPPORT = "applied_t1"

FORBIDDEN_FEATURES = {
    "applied_t1",
    "growth_t1",
    "absolute_change_t1",
    "criticality",
    "risk_score",
    "alert_level",
}

leakage = FORBIDDEN_FEATURES.intersection(FEATURES)
assert not leakage, f"Target leakage détecté : {sorted(leakage)}"

print("Nombre de features :", len(FEATURES))
for i, feature in enumerate(FEATURES, 1):
    print(f"{i:02d}. {feature}")

print("\nOutcome futur conservé hors X :", TARGET_SUPPORT)


Nombre de features : 19
01. year
02. coo_id
03. coa_id
04. coa_name
05. coo_name
06. origin_region
07. asylum_region
08. applied
09. applied_lag1
10. applied_lag2
11. growth_past_1
12. absolute_change_past_1
13. acceleration
14. two_consecutive_increases
15. origin_applied_total_t
16. asylum_applied_total_t
17. segment_share_origin_t
18. segment_share_asylum_t
19. idmc_total_origin_lag1

Outcome futur conservé hors X : applied_t1


## Définition métier retenue pour l'hausse significative des demandes

La cible supervisée sera construite dans le notebook Machine Learning à partir de `applied` et `applied_t1`.

**hausse significative des demandes = hausse d'au moins 30 % ET augmentation d'au moins 100 demandes entre T et T+1.**

Formellement :

`applied_t1 >= applied × 1.30` **ET** `(applied_t1 - applied) >= 100`

`applied_t1` reste uniquement un support de construction de la cible et n'entre jamais dans les variables explicatives du modèle.

Les lignes 2025 restent non labellisées, car T+1 = 2026 n'est pas disponible.

# 13. Data Quality

Aucune imputation statistique n'est réalisée ici.

Les valeurs manquantes restent visibles. Les médianes, encodages et autres transformations apprises devront être ajustés **après le split temporel**, dans le pipeline ML.


In [56]:
final_columns = FEATURES + [TARGET_SUPPORT]
dataset_ml = corridor[final_columns].copy()

dq = pd.DataFrame({
    "column": dataset_ml.columns,
    "dtype": [str(dataset_ml[c].dtype) for c in dataset_ml.columns],
    "missing_n": [int(dataset_ml[c].isna().sum()) for c in dataset_ml.columns],
    "missing_pct": [
        float(dataset_ml[c].isna().mean() * 100)
        for c in dataset_ml.columns
    ],
    "n_unique": [
        int(dataset_ml[c].nunique(dropna=True))
        for c in dataset_ml.columns
    ],
}).sort_values("missing_pct", ascending=False)

display(dq)


,column,dtype,missing_n,missing_pct,n_unique
18,idmc_total_origin_lag1,float64,55278,64.1023,500
13,two_consecutive_increases,Int8,29220,33.8845,2
12,acceleration,float64,29220,33.8845,42689
9,applied_lag2,float64,24406,28.3021,4228
10,growth_past_1,float64,19277,22.3543,23523
8,applied_lag1,float64,19277,22.3543,4429
11,absolute_change_past_1,float64,19277,22.3543,4414
19,applied_t1,float64,3967,4.6003,4536
7,applied,int64,0,0.0000,4622
6,asylum_region,string,0,0.0000,22


In [57]:
# ============================================================
# 14. CONTRÔLES FINAUX
# ============================================================

assert dataset_ml.duplicated(KEY).sum() == 0, (
    "La clé corridor × année n'est pas unique."
)

assert not FORBIDDEN_FEATURES.intersection(FEATURES)

# Contrôle du type nullable de l'indicateur de persistance.
assert str(dataset_ml["two_consecutive_increases"].dtype) == "Int8"

# applied_t1 reste manquant uniquement lorsque T+1 n'est pas encore
# couvert par la source (ex. dernière année observée).
max_year = int(corridor["year"].max())
assert dataset_ml.loc[
    dataset_ml["year"] < max_year, "applied_t1"
].notna().all()

print("Clé corridor × année unique : OK")
print("Anti-leakage : OK")
print("two_consecutive_increases : Int8 nullable OK")
print("Outcome T+1 : 0 pour corridor absent si T+1 observable ; NA sinon")
print("Lignes finales :", len(dataset_ml))
print("Avec outcome T+1 :", int(dataset_ml["applied_t1"].notna().sum()))
print("Sans outcome T+1 :", int(dataset_ml["applied_t1"].isna().sum()))


Clé corridor × année unique : OK
Anti-leakage : OK
two_consecutive_increases : Int8 nullable OK
Outcome T+1 : 0 pour corridor absent si T+1 observable ; NA sinon
Lignes finales : 86234
Avec outcome T+1 : 82267
Sans outcome T+1 : 3967


In [58]:
# ============================================================
# 15. EXPORTS
# ============================================================

dataset_ml.to_csv(OUTPUT_PATH, index=False)
dq.to_csv(DQ_PATH, index=False)
pd.DataFrame(audit).to_csv(AUDIT_PATH, index=False)

print("Dataset ML :", OUTPUT_PATH)
print("Rapport Data Quality :", DQ_PATH)
print("Journal d'audit :", AUDIT_PATH)


Dataset ML : ..\data\feature_engineering_outputs\dataset_ml_features.csv
Rapport Data Quality : ..\data\feature_engineering_outputs\data_quality_feature_engineering.csv
Journal d'audit : ..\data\feature_engineering_outputs\audit_feature_engineering.csv


# 16. Contrat avec le notebook Machine Learning

Le notebook ML recevra :

**X :** uniquement les features listées dans `FEATURES`.

**Outcome historique :** `applied_t1`.

### Sémantique de `applied_t1`

- si T+1 est dans la période couverte par la source et que le corridor est absent en T+1 : **`applied_t1 = 0`** ;
- si T+1 n'est pas encore couvert par la source (ex. 2025 → 2026) : **`applied_t1 = NA`**.

Le notebook ML devra ensuite :
1. construire et justifier la définition d'une **hausse significative des demandesment significative à T+1** ;
2. effectuer un split strictement temporel ;
3. établir une baseline de classification ;
4. comparer Logistic Regression, Random Forest et HistGradientBoosting ;
5. suivre Recall, Precision, F1, PR-AUC et matrice de confusion ;
6. calibrer les probabilités ;
7. transformer la probabilité calibrée en score de risque ;
8. définir les niveaux **Normal / Attention / High / Critical** à partir des performances et de la capacité opérationnelle.

Aucune décision automatique ne doit être déclenchée par le score sans validation du dispositif métier.
